<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/08-window-functions/04-dedup-patterns.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 8.4 — Deduplication: pick the survivor deliberately

We simulate a CDC feed on top of orders.csv (same order_id, multiple
versions), then run the survivor-selection lineup: dropDuplicates
(the trap), window row_number (the standard), max_by (the aggregate).

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("8.4-dedup-patterns")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)

## Build the CDC feed

Version 1 = the original row. Some orders then get corrections:
v2 fixes quantity, v3 fixes country. Newest feed_seq = truth.

In [ ]:
v1 = orders.withColumn("feed_seq", F.lit(1)).withColumn("op", F.lit("insert"))

v2 = (orders.filter(col("order_id").isin(1001, 1012, 1032, 1040))
      .withColumn("quantity", F.coalesce(col("quantity"), F.lit(1)) + 1)
      .withColumn("feed_seq", F.lit(2))
      .withColumn("op", F.lit("update")))

v3 = (orders.filter(col("order_id").isin(1010, 1012))
      .withColumn("country", F.lit("IN"))
      .withColumn("feed_seq", F.lit(3))
      .withColumn("op", F.lit("update")))

updates = v1.unionByName(v2).unionByName(v3)
print("orders:", orders.count(), "| update feed:", updates.count())
updates.filter(col("order_id") == 1012).orderBy("feed_seq").show()

## Trap first: dropDuplicates(subset)

One row per key — but WHICH row is whatever execution met first.
On this small local run it may even look right. Nothing promises it.

In [ ]:
survivors = updates.dropDuplicates(["order_id"])
survivors.filter(col("order_id") == 1012).show()
# run the cell a few times / change shuffle.partitions: the feed_seq
# that survives is not a function of the data

## The standard: window + row_number, policy in one line

"Newest by feed_seq; ties can't happen (order_id+feed_seq unique in
a sane feed) — but we tiebreak anyway, because feeds aren't sane."

In [ ]:
w = Window.partitionBy("order_id").orderBy(F.desc("feed_seq"), F.desc("op"))

latest_w = (updates
            .withColumn("rn", F.row_number().over(w))
            .filter(col("rn") == 1)
            .drop("rn"))

print("keys:", latest_w.count())
latest_w.filter(col("order_id").isin(1012, 1010, 1001)).orderBy("order_id").show()
# 1012 came out with feed_seq 3: quantity fix AND country fix?  NO —
# look closely: v3 was built from the ORIGINAL row, so its quantity is
# still the old one. CDC reality: versions are whole-row snapshots;
# merging PARTIAL updates is a different (harder) problem — Module 15's
# MERGE. The survivor pattern assumes whole-row versions.

## The aggregate: groupBy + max_by (Spark 3.3+)

max_by(payload, ordering): pack the row as a struct, take the one
with max feed_seq, unpack. Map-side partial aggregation collapses
versions BEFORE the shuffle — the physics win on hot keys.

In [ ]:
latest_a = (updates.groupBy("order_id")
            .agg(F.max_by(F.struct(*[c for c in updates.columns]), "feed_seq").alias("row"))
            .select("row.*"))

# same survivors as the window version?
diff = latest_w.exceptAll(latest_a)
print("rows:", latest_a.count(), "| disagreements vs window:", diff.count())

In [ ]:
# pre-3.3 spelling: max of a struct — comparison is field-by-field,
# so ordering columns go FIRST in the struct
latest_s = (updates
            .groupBy("order_id")
            .agg(F.max(F.struct("feed_seq", *[c for c in updates.columns if c != "feed_seq"]))
                 .alias("row"))
            .select("row.*"))
print(latest_s.exceptAll(latest_a.select(latest_s.columns)).count(), "disagreements")

## The physics: two plans

In [ ]:
latest_w.explain()   # Exchange + Sort + Window (+ WindowGroupLimit on 3.5+)

In [ ]:
latest_a.explain()   # HashAggregate (partial) -> Exchange -> HashAggregate (final)
# the 'partial_max_by' before the Exchange is 2.2's map-side combine,
# reborn — per input partition, only ONE candidate per key ships

## The report: dedup observability

Rows in vs out, and who got trimmed — a metric, not a mystery.

In [ ]:
(updates.groupBy("order_id").count()
 .filter(col("count") > 1)
 .orderBy(F.desc("count"), "order_id")
 .show())

report = updates.groupBy().agg(
    F.count("*").alias("rows_in"),
    F.countDistinct("order_id").alias("keys"),
)
report.withColumn("rows_dropped", col("rows_in") - col("keys")).show()

## Exercises

1. Break the feed: add a second row with feed_seq=3 for order 1012
   (different country). Rerun the window and max_by versions several
   times — which one is at least CONSISTENT within a run, and what
   orderBy addition restores full determinism?
2. "Keep the latest 2 versions per key" for an audit table — adapt
   the window pattern (two-token change), then explain why max_by
   can't follow.
3. Route the LOSERS (rn > 1) to a quarantine DataFrame in the same
   pass and write both outputs. Why is this cheaper than running the
   dedup twice with opposite filters? Check the plan to be sure Spark
   agrees (hint: it may not — .cache() the ranked frame and re-check).
4. Delete handling: treat op='delete' rows (add a few) as versions
   that REMOVE the key if they're newest. One added filter — where?
5. Benchmark at scale: rebuild `updates` with spark.range —
   1M keys x 1 version + 100 keys x 10_000 versions (hot keys). Time
   window vs max_by. Relate the winner's margin to the partial_max_by
   line in its plan.

In [ ]:
# your exercise code here